# BGG Success Predictor — Pre-processing

Loads the four source CSVs, applies all pre-processing required for all planned algorithms, computes the two success score target labels, and saves a single clean CSV ready for training.

**Input:** `dataset/`  
**Output:** `data/games_processed.csv`

This notebook is designed to be fully re-runnable and deterministic — running it twice produces identical output.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("dataset")
OUT_DIR  = Path("data")
OUT_DIR.mkdir(exist_ok=True)

OUT_PATH = OUT_DIR / "games_processed.csv"
if OUT_PATH.exists():
    OUT_PATH.unlink()
assert not OUT_PATH.exists(), "Failed to clear games_processed.csv"
print("Output path cleared and ready.")

## 2. Load source CSVs

In [2]:
games         = pd.read_csv(DATA_DIR / "games.csv")
mechanics     = pd.read_csv(DATA_DIR / "mechanics.csv")
themes        = pd.read_csv(DATA_DIR / "themes.csv")
subcategories = pd.read_csv(DATA_DIR / "subcategories.csv")

print(f"games:         {games.shape}")
print(f"mechanics:     {mechanics.shape}")
print(f"themes:        {themes.shape}")
print(f"subcategories: {subcategories.shape}")

games:         (21925, 48)
mechanics:     (21925, 158)
themes:        (21925, 218)
subcategories: (21925, 11)


## 3. Drop unused columns from games.csv

Keep only columns with role `feature`, `quality_score`, or `commercial_score` per the pre-processing plan.

In [3]:
KEEP_COLS = [
    # identifier — kept in output for game lookups
    "BGGId",
    # features
    "GameWeight", "MinPlayers", "MaxPlayers",
    "ComAgeRec", "MfgAgeRec",
    "MfgPlaytime", "ComMinPlaytime", "ComMaxPlaytime",
    # binary category flags
    "Cat:Thematic", "Cat:Strategy", "Cat:War", "Cat:Family",
    "Cat:CGS", "Cat:Abstract", "Cat:Party", "Cat:Childrens",
    # success score inputs
    "BayesAvgRating", "NumOwned", "YearPublished",
]

games = games[KEEP_COLS].copy()
print(games.shape)

(21925, 20)


## 4. Join all four sources on BGGId

In [4]:
df = (
    games
    .merge(mechanics,     on="BGGId", how="left")
    .merge(themes,        on="BGGId", how="left")
    .merge(subcategories, on="BGGId", how="left")
    .copy()
)

print(f"Joined shape: {df.shape}")
assert len(df) == len(games), "Row count changed after join — unexpected BGGId mismatch"

Joined shape: (21925, 404)


## 5. Feature pre-processing

### 5a. GameWeight — impute 0 (unrated) with median of rated games

In [5]:
median_weight = df.loc[df["GameWeight"] > 0, "GameWeight"].median()
print(f"GameWeight median (excl zeros): {median_weight:.4f}")
df["GameWeight"] = df["GameWeight"].replace(0, median_weight)

GameWeight median (excl zeros): 2.0000


### 5b. MinPlayers — impute 0 with 1 (a game always needs at least 1 player)

In [6]:
df["MinPlayers"] = df["MinPlayers"].replace(0, 1)

### 5c. MaxPlayers — cap at 20, impute 0 with median

Max value in dataset is 999, a sentinel for "unlimited". p95 is 10 so cap at 20 is generous.

In [7]:
df["MaxPlayers"] = df["MaxPlayers"].clip(upper=20)
median_maxplayers = df.loc[df["MaxPlayers"] > 0, "MaxPlayers"].median()
print(f"MaxPlayers median (excl zeros, post-cap): {median_maxplayers}")
df["MaxPlayers"] = df["MaxPlayers"].replace(0, median_maxplayers)

MaxPlayers median (excl zeros, post-cap): 4.0


### 5d. ComAgeRec — impute NaN with MfgAgeRec first, then median for remaining

25% missing. MfgAgeRec (manufacturer stated) is a reasonable proxy.

In [8]:
missing_before = df["ComAgeRec"].isna().sum()

# Fill from MfgAgeRec where MfgAgeRec > 0
mask = df["ComAgeRec"].isna() & (df["MfgAgeRec"] > 0)
df.loc[mask, "ComAgeRec"] = df.loc[mask, "MfgAgeRec"]

# Fill remaining with median
median_age = df["ComAgeRec"].median()
df["ComAgeRec"] = df["ComAgeRec"].fillna(median_age)

print(f"ComAgeRec missing: {missing_before} → {df['ComAgeRec'].isna().sum()}")

ComAgeRec missing: 5530 → 0


### 5e. MfgAgeRec — impute 0 with median

In [9]:
median_mfgage = df.loc[df["MfgAgeRec"] > 0, "MfgAgeRec"].median()
print(f"MfgAgeRec median (excl zeros): {median_mfgage}")
df["MfgAgeRec"] = df["MfgAgeRec"].replace(0, median_mfgage)

MfgAgeRec median (excl zeros): 10.0


### 5f. Playtime columns — cap at 600 min, impute 0 with median

Max in dataset is 60,000 min — clearly sentinel/error values. p95 is 240 so 600 (10 hrs) is a generous cap.

In [10]:
PLAYTIME_COLS = ["MfgPlaytime", "ComMinPlaytime", "ComMaxPlaytime"]
PLAYTIME_CAP  = 600

for col in PLAYTIME_COLS:
    df[col] = df[col].clip(upper=PLAYTIME_CAP)
    median_pt = df.loc[df[col] > 0, col].median()
    df[col] = df[col].replace(0, median_pt)
    print(f"{col}: capped at {PLAYTIME_CAP}, 0s imputed with median {median_pt}")

MfgPlaytime: capped at 600, 0s imputed with median 45.0
ComMinPlaytime: capped at 600, 0s imputed with median 30.0
ComMaxPlaytime: capped at 600, 0s imputed with median 45.0


## 6. Compute success score target labels

### 6a. quality_score = BayesAvgRating / 10

Actual range in dataset is 3.57–8.51, so output will be 0.357–0.851.

In [11]:
df["quality_score"] = df["BayesAvgRating"] / 10
print(f"quality_score range: {df['quality_score'].min():.3f} – {df['quality_score'].max():.3f}")

quality_score range: 0.357 – 0.851


### 6b. commercial_score = log1p(NumOwned / years_active), normalised to 0–1

`years_active = clamp(2025 − YearPublished, 1, 10)` — ancient games (pre-2015) all get 10.

In [12]:
years_active = (2025 - df["YearPublished"]).clip(lower=1, upper=10)
raw_commercial = np.log1p(df["NumOwned"] / years_active)

df["commercial_score"] = raw_commercial / raw_commercial.max()
df = df.copy()
print(f"commercial_score range: {df['commercial_score'].min():.3f} – {df['commercial_score'].max():.3f}")

commercial_score range: 0.000 – 1.000


## 7. Drop score input columns (not training features)

In [13]:
df = df.drop(columns=["BayesAvgRating", "NumOwned", "YearPublished"])
print(f"Final shape: {df.shape}")

Final shape: (21925, 403)


## 8. Sanity check

In [14]:
print("Missing values per column (should all be 0):")
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else "  None")

print(f"\nRows: {len(df):,}")
print(f"Cols: {len(df.columns):,}")
print(f"  — continuous features: 8")
print(f"  — binary (Cat + mechanics + themes + subcategories): {len(df.columns) - 8 - 2}")
print(f"  — target labels (quality_score, commercial_score): 2")

df.describe().T[["min", "50%", "max"]].rename(columns={"50%": "median"})

# Assertions
assert len(df) == 21_925, f"Expected 21,925 rows, got {len(df)}"
assert len(df.columns) == 403, f"Expected 403 columns, got {len(df.columns)}"
missing = df.isna().sum().sum()
assert missing == 0, f"Found {missing} missing values"
assert df["quality_score"].between(0, 1).all(), "quality_score out of 0–1 range"
assert df["commercial_score"].between(0, 1).all(), "commercial_score out of 0–1 range"
pandemic = df[df["BGGId"] == 30549].iloc[0]
assert pandemic["quality_score"] > 0.7, f"Pandemic quality_score suspiciously low: {pandemic['quality_score']:.3f}"
assert pandemic["commercial_score"] > 0.7, f"Pandemic commercial_score suspiciously low: {pandemic['commercial_score']:.3f}"
print(f"Pandemic — quality: {pandemic['quality_score']:.3f}, commercial: {pandemic['commercial_score']:.3f}")
print("All assertions passed.")

Missing values per column (should all be 0):
  None

Rows: 21,925
Cols: 403
  — continuous features: 8
  — binary (Cat + mechanics + themes + subcategories): 393
  — target labels (quality_score, commercial_score): 2


Pandemic — quality: 0.749, commercial: 1.000
All assertions passed.


## 9. Save

In [15]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}  ({OUT_PATH.stat().st_size / 1e6:.1f} MB)")

Saved: data/games_processed.csv  (18.6 MB)
